In [1]:
from ash import *

ashpath: /Users/tom/Documents/ucl/projects/ash-fork/ash
Sys path: ['/Users/tom/Documents/ucl/projects/ash-fork/ash', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python311.zip', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11/lib-dynload', '', '/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages', '__editable__.ash-0.95.finder.__path_hook__']
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                                           ASH                                            
                              A MULTISCALE MODELLING PROGRAM                              
                                        Version: 0.9dev                                        
                               Git c

In [97]:
BASIS = 'sto-3g'
XC = 'b3lyp'
CHARGE = -3
MULT = 1

In [98]:
#Defining fragment
frag = Fragment(
    xyzfile="system_aftersolvent.xyz", 
    charge=CHARGE, 
    mult=MULT
)


--------------------------------------------------------------------------------
                                New ASH fragment                                
--------------------------------------------------------------------------------

ASH Fragment creation
Reading coordinates from XYZ file 'system_aftersolvent.xyz' into fragment.
Creating/Updating fragment attributes...
Number of Atoms in fragment: 2621
Formula: Na3P1O875H1742
Label: system_aftersolvent
Charge: -3 Mult: 1

--------------------------------------------------------------------------------


In [99]:
xyz_list = [i for i in zip(frag.elems, frag.coords)]

lines = [str(len(xyz_list)), '']
for symbol, coords in xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

xyz_string = "\n".join(lines)

In [100]:
qm_atoms = [0,1,2,3,4]

In [101]:
qm_xyz_list = [
    (frag.elems[i], frag.coords[i])
    for i in qm_atoms
]

lines = [str(len(qm_xyz_list)), '']
for symbol, coords in qm_xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

qm_xyz_string = "\n".join(lines)
print(qm_xyz_string)

5

P 0.0 0.0 0.0
O 0.0 0.0 1.713
O 0.0 1.615 -0.571
O 1.399 -0.807 -0.571
O -1.399 -0.807 -0.571


In [102]:
N_ACT = 2

In [103]:
nbed_theory = NbedTheory(
    geometry=qm_xyz_string,
    n_active_atoms=N_ACT,
    basis=BASIS,
    xc_functional=XC,
    projector='mu',
    localization='spade',
    # run_ccsd_emb=True
)



                     #####################################                      
                     #                                   #                      
                     #     NbedTheory initialization     #                      
                     #                                   #                      
                     #####################################                      


In [104]:
import nbed

In [105]:
MULT

1

In [108]:
import pyscf

mol = pyscf.M(
    atom="PO4.xyz",
    charge=CHARGE,
    spin = 0.5 * (MULT - 1),
    basis=BASIS,
)

rhf_obj = pyscf.scf.RHF(mol).run()

converged SCF energy = -630.835276933496


In [109]:
mol.spin

0

In [110]:
nbed_config = nbed.config.NbedConfig(
    geometry = qm_xyz_string,
    n_active_atoms = N_ACT,
    basis = BASIS,
    xc_functional = XC,
    projector = 'mu',
    localization = 'spade',
    spin = 0.5 * (MULT - 1),
    # run_ccsd_emb=True
)

nbed_result = nbed.nbed(nbed_config)

RuntimeError: Electron number 47 and spin 0 are not consistent
Note mol.spin = 2S = Nalpha - Nbeta, not 2S+1

In [73]:
nbed_result.embedded_scf.mo_coeff.shape

(2, 29, 18)

In [75]:
# water_xml = "/opt/homebrew/Caskroom/miniconda/base/envs/ash-conda/lib/python3.11/site-packages/openmm/app/data/amber14/tip3p.xml"
water_xml = "amber14/tip3p.xml"

frozen_atoms=listdiff(frag.allatoms,qm_atoms)

openmm_theory = OpenMMTheory(
    xmlfiles=["openff_LIG.xml", water_xml], 
    pdbfile="system_aftersolvent.pdb", 
    # periodic=True, 
    # autoconstraints=None,
    # rigidwater=False,
    # frozen_atoms=qm_atoms,
)



                           #########################                            
                           #                       #                            
                           #     OpenMM Theory     #                            
                           #                       #                            
                           #########################                            
OpenMM CPU threads set to: 1
Imported OpenMM library version: 8.3.1

--------------------------------------------------------------------------------
                             Defining OpenMM object                             
--------------------------------------------------------------------------------

Printlevel: 2
HBonds option: X-H bond lengths will automatically be constrained
AutoConstraint setting: HBonds
Rigidwater constraints: True
Hydrogenmass option: 1.5 Da
Using platform: CPU

--------------------------------------------------------------------------------
          

In [76]:
qmmm_theory = QMMMTheory(
    qm_theory = nbed_theory,
    mm_theory = openmm_theory,
    fragment = frag,
    qm_charge = CHARGE,
    qm_mult = MULT,
    qmatoms = qm_atoms,
    printlevel = 3,
)



                            ########################                            
                            #                      #                            
                            #     QM/MM Theory     #                            
                            #                      #                            
                            ########################                            
QM-theory: NbedTheory
MM-theory: OpenMMTheory
All atoms in fragment: 2621
QM region (5 atoms): [0, 1, 2, 3, 4]
MM region (2616 atoms)
QM/MM object selected to use 1 cores
Embedding: elstat
No atomcharges list passed to QMMMTheory object
Getting system charges from OpenMM object
QM-region coordinates (before linkatoms):
   0    P   0.00000000    0.00000000    0.00000000
   1    O   0.00000000    0.00000000    1.71300000
   2    O   0.00000000    1.61500000   -0.57100000
   3    O   1.39900000   -0.80700000   -0.57100000
   4    O  -1.39900000   -0.80700000   -0.57100000

Determining QM-M

In [77]:
qmmm_theory

In [78]:
MolecularDynamics(
    fragment=frag,
    theory=qmmm_theory,
    timestep=0.001,
    simulation_steps=10,
    traj_frequency=1,
    temperature=300,
    # integrator='LangevinIntegrator',
    coupling_frequency=1,
    charge=CHARGE,
    mult=MULT
)



                     ######################################                     
                     #                                    #                     
                     #     OpenMM MD wrapper function     #                     
                     #                                    #                     
                     ######################################                     


              ####################################################              
              #                                                  #              
              #     OpenMM Molecular Dynamics Initialization     #              
              #                                                  #              
              ####################################################              
Analyzing theory input to OpenMM_MDclass
This is an QMMMTheory object
Turning on externalforce option.
Added force
System is non-periodic. Setting enforcePeriodicBox to False

----------

RuntimeError: Electron number 47 and spin 0 are not consistent
Note mol.spin = 2S = Nalpha - Nbeta, not 2S+1

In [13]:
# Optimizer(
#     fragment=frag, 
#     theory=qmmm_theory, 
#     ActiveRegion=True, 
#     actatoms=qm_atoms
# )